In [28]:
import re
from pathlib import Path

import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [29]:
BASE_DIR = Path.cwd()
PDF_PATH = BASE_DIR / "RAG_Notes_Complete.pdf"
FAISS_DIR = BASE_DIR / "faiss_index"

EMBED_MODEL = "nomic-embed-text"
OLLAMA_BASE_URL = "http://localhost:11434"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 80

print("Working folder :", BASE_DIR)
print("PDF exists     :", PDF_PATH.exists())


Working folder : c:\Users\ARTI\Desktop\Abhishek\RAG-Chatbot-2-FAISS
PDF exists     : True


In [30]:
loader = PyPDFLoader(str(PDF_PATH))
documents = loader.load()

print("Pages loaded:", len(documents))
print("Type of each item:", type(documents[0]))


Pages loaded: 9
Type of each item: <class 'langchain_core.documents.base.Document'>


In [31]:
doc = documents[0]
print("page_content (first 300 chars):\n", doc.page_content[:300])
print("\nmetadata:", doc.metadata)


page_content (first 300 chars):
 RAG Notes (updated)
Page 1
 RAG Notes (updated)

metadata: {'producer': '', 'creator': '', 'creationdate': '2026-11-15T15:48:22+00:00', 'moddate': '2026-11-15T15:48:22+00:00', 'source': 'c:\\Users\\ARTI\\Desktop\\Abhishek\\RAG-Chatbot-2-FAISS\\RAG_Notes_Complete.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}


In [32]:
for d in documents:
    text = re.sub(r"[ \t]+", " ", d.page_content)   # many spaces/tabs -> one space
    text = re.sub(r"\n{3,}", "\n\n", text)          # 3+ blank lines -> one blank line
    d.page_content = text.strip()

empty_pages = [d.metadata["page"] + 1 for d in documents if not d.page_content]
print("Pages with no text:", empty_pages)
print("Total characters  :", sum(len(d.page_content) for d in documents))


Pages with no text: [9]
Total characters  : 7841


In [33]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
    is_separator_regex=False,
)

chunks = splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i

print("Total chunks:", len(chunks))


Total chunks: 20


In [34]:
sizes = [len(c.page_content) for c in chunks]
print(f"Chunk sizes -> min {min(sizes)}, max {max(sizes)}, average {sum(sizes)/len(sizes):.0f}")

for chunk in chunks[:5]:
    print(f"chunk {chunk.metadata['chunk_id']} | page {chunk.metadata['page'] + 1} | {len(chunk.page_content)} chars")

print("\n=== Chunk 0 ===")
print(chunks[0].page_content)


Chunk sizes -> min 47, max 500, average 424
chunk 0 | page 1 | 47 chars
chunk 1 | page 2 | 485 chars
chunk 2 | page 2 | 476 chars
chunk 3 | page 2 | 428 chars
chunk 4 | page 3 | 497 chars

=== Chunk 0 ===
RAG Notes (updated)
Page 1
 RAG Notes (updated)


In [35]:
for n in range(len(chunks) - 1):
    a, b = chunks[n], chunks[n + 1]
    if a.metadata["page"] == b.metadata["page"]:
        print(f"End of chunk {n}   : ...{a.page_content[-80:]!r}")
        print(f"Start of chunk {n+1}: {b.page_content[:80]!r}...")
        break
    

End of chunk 1   : ...' relevant parts of your documents.\n \x7f Give the retrieved information to the LLM.'
Start of chunk 2: '\x7f Give the retrieved information to the LLM.\n \x7f The LLM answers based on that do'...


In [36]:
embeddings = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_BASE_URL)

chunk_vectors = embeddings.embed_documents([c.page_content for c in chunks[:3]])

print("Vectors made (test on 3 chunks):", len(chunk_vectors))
print("Dimensions per vector          :", len(chunk_vectors[0]))
print("First 5 numbers of chunk 0     :", chunk_vectors[0][:5])


Vectors made (test on 3 chunks): 3
Dimensions per vector          : 768
First 5 numbers of chunk 0     : [0.013658359, -0.042788927, -0.20990323, -0.061483257, -0.015921947]


In [37]:
vector_store = FAISS.from_documents(
    chunks,
    embeddings,
    distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT,
    normalize_L2=True,
)
print("Vectors in index:", vector_store.index.ntotal)


Vectors in index: 20


In [38]:
vector_store.save_local(str(FAISS_DIR))

for f in FAISS_DIR.iterdir():
    print(f.name, "-", f.stat().st_size, "bytes")
    

index.faiss - 61485 bytes
index.pkl - 11684 bytes


In [39]:
loaded = FAISS.load_local(
    str(FAISS_DIR),
    embeddings,
    allow_dangerous_deserialization=True,
    normalize_L2=True,
    distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT,
)
print("Reloaded vectors:", loaded.index.ntotal)

question = "What is chunking in RAG?"
results = loaded.similarity_search_with_score(question, k=3)

for doc, score in results:
    print(f"\nscore={score:.3f} | chunk {doc.metadata['chunk_id']} | page {doc.metadata['page'] + 1}")
    print(doc.page_content[:150].replace("\n", " "), "...")
    

Reloaded vectors: 20

score=0.608 | chunk 9 | page 4
 A fallback when no relevant chunk is found ("I don't know").   Text splitting Types of text splitters 1. CharacterTextSplitter   Splits on a single ...

score=0.594 | chunk 7 | page 4
RAG Notes (updated) Page 4 Step 3. Find relevant chunks   Compare the query vector with the document vectors in the vector DB.   Take the top-k most ...

score=0.580 | chunk 0 | page 1
RAG Notes (updated) Page 1  RAG Notes (updated) ...


In [40]:
import numpy as np

q = np.array(embeddings.embed_query(question))
top_doc, top_score = results[0]
c = np.array(embeddings.embed_query(top_doc.page_content))

manual_cosine = q @ c / (np.linalg.norm(q) * np.linalg.norm(c))
print(f"FAISS score : {top_score:.4f}")
print(f"Manual cosine: {manual_cosine:.4f}")


FAISS score : 0.6083
Manual cosine: 0.6083
